# Working Backwards Experiments
This notebook recomputes the latency distribution experiment and persists the Population Stability Index (PSI) so downstream reports can consume a numeric value.


In [ ]:
from __future__ import annotations

import json
from datetime import datetime
from pathlib import Path
from math import log

baseline_latency = {"0-200ms": 0.42, "200-400ms": 0.37, "400ms+": 0.21}
current_latency = {"0-200ms": 0.40, "200-400ms": 0.38, "400ms+": 0.22}

def calculate_psi(baseline: dict[str, float], current: dict[str, float], *, epsilon: float = 1e-6) -> float:
    """Compute PSI with a guard for empty buckets."""
    psi = 0.0
    for bucket, baseline_share in baseline.items():
        current_share = current.get(bucket, epsilon)
        baseline_adj = baseline_share if baseline_share > 0 else epsilon
        current_adj = current_share if current_share > 0 else epsilon
        psi += (current_adj - baseline_adj) * log(current_adj / baseline_adj)
    return psi

psi_value = calculate_psi(baseline_latency, current_latency)
psi_value


In [ ]:
artifact_path = Path("docs/notebooks/artifacts/latency_summary.json")
artifact_path.parent.mkdir(parents=True, exist_ok=True)

latency_summary = {
    "experiment_id": "working_backwards_experiments",
    "generated_at": datetime.utcnow().replace(microsecond=0).isoformat() + "Z",
    "baseline_latency_share": baseline_latency,
    "current_latency_share": current_latency,
    "metrics": {
        "psi": psi_value,
        "psi_threshold": 0.2
    }
}

with artifact_path.open("w", encoding="utf-8") as fp:
    json.dump(latency_summary, fp, indent=2, sort_keys=True)

latency_summary
